# Análisis de fuga de clientes (Bank Churners)

Este notebook documenta los hallazgos del análisis exploratorio, reutilizando
las funciones y la clase `AnalizadorClientes` definidas en
`Analisis_TarjetaDeCredito.py`, en vez de duplicar lógica.

**Nota sobre reproducibilidad:** las celdas de este notebook dependen de que
el CSV esté disponible en `data/BankChurners.csv` (o se ajuste la ruta abajo).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from Analisis_TarjetaDeCredito import CargarDatos, AnalizadorClientes, ResolverRutaDatos
import pandas as pd

ruta_datos = ResolverRutaDatos(None)
df = CargarDatos(ruta_datos)
df.shape

## 1. Segmentación y limpieza

In [ ]:
analizador = AnalizadorClientes(df)
analizador.SegmentaCredito()
analizador.SegmentoGenero()
df[["Credit_Limit", "Segmento_Credito", "Gender", "Segmento_Genero"]].head()

## 2. Fuga por segmento de crédito

**Hallazgo clave:** los clientes en el 25% más bajo de límite de crédito
("Credito bajo") tienen una tasa de fuga notablemente más alta que el resto.
Esto es un segmento candidato prioritario para campañas de retención.

In [ ]:
cruce_credito = analizador.CruzarSegmento()
cruce_credito

## 3. Fuga por género

In [ ]:
cruce_genero = analizador.CruzarGenero()
cruce_genero

## 4. Cruce género × segmento de crédito

In [ ]:
cruce_combinado = analizador.CruzarGenero_Credito()
cruce_combinado

## 5. Significancia estadística

Chi-cuadrado sobre las tablas de contingencia (crédito y género), y t-test
para las variables continuas entre clientes fugados y existentes.

In [ ]:
from scipy import stats

for nombre, cruce in [("Segmento de crédito", cruce_credito), ("Género", cruce_genero)]:
    tabla = cruce.drop(columns="%_Fuga")
    chi2, p, dof, _ = stats.chi2_contingency(tabla)
    print(f"{nombre}: chi2={chi2:.2f}, p-value={p:.4f}")

In [ ]:
Clientes_Fugados = df[df["Attrition_Flag"] == "Attrited Customer"]
Clientes_Existentes = df[df["Attrition_Flag"] == "Existing Customer"]

for variable in ["Customer_Age", "Total_Trans_Ct", "Credit_Limit"]:
    t_stat, p_valor = stats.ttest_ind(
        Clientes_Fugados[variable], Clientes_Existentes[variable], equal_var=False
    )
    significativo = "sí" if p_valor < 0.05 else "no"
    print(f"{variable}: t={t_stat:.2f}, p-value={p_valor:.4f} (significativo: {significativo})")

## 6. Conclusiones

- El segmento de menor límite de crédito muestra una fuga significativamente
  mayor (chi-cuadrado, p < 0.05) — candidato prioritario para retención.
- El número de transacciones (`Total_Trans_Ct`) es la variable continua con
  la diferencia más marcada y significativa entre fugados y existentes.
- El género muestra una asociación estadísticamente significativa con la
  fuga, aunque de menor magnitud que el segmento de crédito.

### Próximos pasos
- Modelado predictivo (regresión logística / árboles) usando estas variables
  como features iniciales.
- Profundizar en la interacción género × crédito con un tamaño de muestra
  mayor por celda.